# Online Forum Multi-Label Communicative Type Classification

Casual online forum comments written in **Bahasa Melayu**, **English**, and **Manglish** from Lowyat Kopitiam are classified according to the communicative purpose expressed by the user.

### The 6 Communicative Categories
- **Inquiry**: Questions, seeking help, recommendations, or troubleshooting.
- **Complaint**: Venting frustration, complaining about bad service, high prices, or problems.
- **Opinion**: Personal viewpoints, beliefs, reviews, or arguments.
- **Information**: Objective facts, news, official updates, guides, or links.
- **Expressive**: Jokes, laughing, memes, greetings, or casual banter.
- **Spam**: Unwanted promotional links, referral links, or automated bot posts.

A single forum comment may serve multiple communicative purposes; therefore, **Multi-Label Classification** is used.

# 0.0 Environment Setup & Configuration
The required dependencies are installed and non-critical library warnings are suppressed for clean execution.

## 0.1 Install Required Libraries
The project dependencies are specified in `requirements.txt`.

In [ ]:
pip install -r requirements.txt

## 0.2 Suppress Warnings, Markdown & Diff Export Helper Setup
Configure warning filters, set Pandas column width display options, define `printmd()`, and define `export_step_diff()` to export before/after step text changes to CSV for external verification.

In [ ]:
import warnings
import pandas as pd
from IPython.display import display, Markdown, HTML

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Disable text truncation in DataFrame displays
pd.set_option('display.max_colwidth', None)
def printmd(string):
    display(Markdown(string))

# Reusable helper to export before & after preprocessing text to CSV
def export_step_diff(prev_series, curr_series, step_name, filename=None, sample_size=None):
    """
    Exports all rows (Row ID, Before, After, have_modified) of any preprocessing step to a CSV file.
    Useful for inspecting diffs or validating logic in external LLMs/tools.
    """
    diff_mask = prev_series != curr_series
    total_affected = diff_mask.sum()
    
    export_df = pd.DataFrame({
        "row_id": prev_series.index,
        "before": prev_series.values,
        "after": curr_series.values,
        "have_modified": diff_mask.values
    })
    
    if sample_size and len(export_df) > sample_size:
        export_df = export_df.head(sample_size)
        
    if filename is None:
        safe_name = "".join([c if c.isalnum() else "_" for c in step_name]).strip("_").lower()
        filename = f"diff_{safe_name}.csv"
        
    export_df.to_csv(filename, index=False, encoding="utf-8")
    printmd(f"**[Export Diff]** Exported `{len(export_df):,}` rows (`{total_affected:,}` modified) from `[{step_name}]` to **`{filename}`**")
    return export_df

# Step 1: Load Dataset & Exploratory Data Analysis (EDA)
The annotated forum dataset is loaded to inspect total records, target intent distributions, average word count per communicative category, and multi-label intent combination co-occurrences.

### 1.1 Load Dataset & Preview Records
The raw CSV dataset is loaded into Pandas to inspect record count, schema, and sample forum comments.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load dataset
df = pd.read_csv("dataset.csv", encoding="utf-8")

# Target intent columns
target_cols = ['Inquiry', 'Complaint', 'Opinion', 'Information', 'Expressive', 'Spam']

# Display basic summary and preview of records
printmd(f"**Dataset Summary:** `{len(df):,}` total rows, `{df.shape[1]}` columns")
printmd("#### Preview of First 5 Records:")
display(df.head())

### 1.2 Individual Intent Distribution & Average Post Length (Word Count)
Visualizes the overall distribution of the 6 communicative intent classes alongside the average post length (word count) for each intent category.

In [ ]:
# 1. Compute Individual Target Intent Distribution
intent_counts = df[target_cols].sum().sort_values(ascending=False)

# 2. Compute Average Word Counts per Intent
df['word_count'] = df['text'].astype(str).apply(lambda x: len(x.split()))
df['char_length'] = df['text'].astype(str).apply(len)

avg_word_counts = {}
for col in target_cols:
    avg_word_counts[col] = df[df[col] == 1]['word_count'].mean()
avg_words_series = pd.Series(avg_word_counts).sort_values(ascending=False)

# 3. 2 Side-by-Side Plots (Intent Distribution & Average Word Count)
fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

# Subplot 1: Distribution of Individual Intent Categories
sns.barplot(x=intent_counts.index, y=intent_counts.values, ax=axes[0], palette="viridis")
axes[0].set_title("Distribution of Individual Intent Categories", fontsize=12, fontweight='bold')
axes[0].set_xlabel("Intent Category", fontsize=11, fontweight='bold')
axes[0].set_ylabel("Total Post Count", fontsize=11, fontweight='bold')
axes[0].set_ylim(0, intent_counts.values[0] * 1.12)
axes[0].tick_params(axis='x', rotation=25)
for i, v in enumerate(intent_counts.values):
    axes[0].text(i, v + 400, f"{v:,}", ha='center', fontsize=9, fontweight='bold')

# Subplot 2: Average Post Length (Word Count) vs Intent Category
sns.barplot(x=avg_words_series.index, y=avg_words_series.values, ax=axes[1], palette="mako")
axes[1].set_title("Average Post Length (Word Count) vs Intent Category", fontsize=12, fontweight='bold')
axes[1].set_xlabel("Intent Category", fontsize=11, fontweight='bold')
axes[1].set_ylabel("Average Words per Post", fontsize=11, fontweight='bold')
axes[1].set_ylim(0, avg_words_series.values[0] * 1.12)
axes[1].tick_params(axis='x', rotation=25)
for i, v in enumerate(avg_words_series.values):
    axes[1].text(i, v + 0.6, f"{v:.1f}", ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

### 1.3 Top 10 Multi-Label Intent Combinations Visualization
Visualizes the 10 most common co-occurring communicative intent combinations found in the Lowyat Kopitiam dataset.

In [ ]:
# Format Multi-Label Intent Combinations
def format_combo(row):
    active = [col for col in target_cols if row[col] == 1]
    if len(active) == 0:
        return 'None (0 Intents)'
    elif len(active) == 1:
        return f"{active[0]} (Only)"
    else:
        return ' + '.join(active)
    
df['intent_combination'] = df[target_cols].apply(format_combo, axis=1)
all_combinations = df['intent_combination'].value_counts()
top_10_combinations = all_combinations.head(10)

# Plot Dedicated Top 10 Intent Combinations Graph (Increased Height & Clean Axis Label)
plt.figure(figsize=(15, 6.8))
sns.barplot(x=top_10_combinations.index, y=top_10_combinations.values, palette="magma")
plt.title("Top 10 Assigned Multi-Label Intent Combinations in Dataset", fontsize=13, fontweight='bold')
plt.xlabel("Intent Combination", fontsize=11, fontweight='bold')
plt.ylabel("Total Post Count", fontsize=11, fontweight='bold')
plt.ylim(0, top_10_combinations.values[0] * 1.15)
plt.xticks(rotation=30, ha='right', fontsize=9.5)
for i, v in enumerate(top_10_combinations.values):
    plt.text(i, v + 350, f"{v:,}\n({v/len(df)*100:.2f}%)", ha='center', fontsize=8.5, fontweight='bold')
plt.tight_layout()
plt.show()

### 1.4 Complete Breakdown of All Assigned Intent Combinations Table
Displays all unique communicative intent combinations present in the dataset with 4 decimal point precision for percentages.

In [ ]:
# Display complete breakdown table with 4 decimal place precision
combo_df = pd.DataFrame({
    "Intent Combination": all_combinations.index,
    "Total Posts": all_combinations.values,
    "Percentage (%)": (all_combinations.values / len(df) * 100).round(4)
})

printmd(f"#### Complete Breakdown of All Assigned Intent Combinations in Dataset ({len(all_combinations)} Unique Combinations):")
display(combo_df.style.format({"Percentage (%)": "{:.4f}%"}))

### Preprocessing Change Inspection Helper (Visual Colored HTML Diff View)
We define `show_step_diff()` using Python's `difflib.SequenceMatcher` and `html.escape()` to render a clean, left-aligned side-by-side HTML diff table:
- **Left-Aligned Text**: All text content is explicitly aligned to the left.
- **HTML Escaped**: Raw HTML/iframe/media tags in posts are safely escaped as plain text to prevent accidental video embeds or formatting distortions.
- **Strict Column Proportions**: Row ID (8%), BEFORE (46%), AFTER (46%).
- **Guaranteed 5-Sample Representation**: Prioritizes concise representative samples (< 250 chars), but always fills up to 5 full samples by pulling from the modified rows if fewer than 5 short posts exist.
- **Visual Highlighting**: Removed tokens in red strikethrough, normalized tokens in bold green.

In [ ]:
import difflib
import html

def show_step_diff(prev_series, curr_series, step_name):
    diff_mask = prev_series != curr_series
    total_affected = diff_mask.sum()
    printmd(f"**[{step_name}]** Rows updated: `{total_affected:,}` ({total_affected/len(prev_series)*100:.2f}%)")
    
    if total_affected == 0:
        printmd("_No rows modified in this step._")
        return
        
    changed_df = pd.DataFrame({
        "before": prev_series[diff_mask],
        "after": curr_series[diff_mask]
    })
    
    # Prioritize concise representative posts (< 250 chars), but always guarantee 5 samples if available
    short_samples = changed_df[changed_df["before"].astype(str).str.len() < 250]
    remaining_samples = changed_df[~changed_df.index.isin(short_samples.index)]
    sample_df = pd.concat([short_samples, remaining_samples]).head(5) if len(changed_df) >= 5 else changed_df
    
    html_rows = []
    for idx, row in sample_df.iterrows():
        b_words = str(row["before"]).split()
        a_words = str(row["after"]).split()
        matcher = difflib.SequenceMatcher(None, b_words, a_words)
        
        b_out, a_out = [], []
        for tag, i1, i2, j1, j2 in matcher.get_opcodes():
            b_chunk = html.escape(" ".join(b_words[i1:i2]))
            a_chunk = html.escape(" ".join(a_words[j1:j2]))
            
            if tag == 'equal':
                b_out.append(b_chunk)
                a_out.append(a_chunk)
            elif tag == 'replace':
                b_out.append(f"<span style='background-color:#ffebee; color:#c62828; text-decoration:line-through;'>{b_chunk}</span>")
                a_out.append(f"<span style='background-color:#e8f5e9; color:#2e7d32; font-weight:bold;'>{a_chunk}</span>")
            elif tag == 'delete':
                b_out.append(f"<span style='background-color:#ffebee; color:#c62828; text-decoration:line-through;'>{b_chunk}</span>")
            elif tag == 'insert':
                a_out.append(f"<span style='background-color:#e8f5e9; color:#2e7d32; font-weight:bold;'>{a_chunk}</span>")
                
        html_rows.append(f'''
        <tr style="border-bottom: 1px solid #e0e0e0;">
            <td style="padding: 8px 6px; font-weight: bold; vertical-align: top; width: 8%; color: #555; text-align: left; word-break: break-word;">Row {idx}</td>
            <td style="padding: 8px 10px; vertical-align: top; width: 46%; font-family: monospace; font-size: 12px; line-height: 1.5; text-align: left; word-break: break-word; white-space: normal;">{' '.join(b_out)}</td>
            <td style="padding: 8px 10px; vertical-align: top; width: 46%; font-family: monospace; font-size: 12px; line-height: 1.5; text-align: left; word-break: break-word; white-space: normal;">{' '.join(a_out)}</td>
        </tr>
        ''')
        
    table_html = f'''
    <table style="table-layout: fixed; width: 100%; border-collapse: collapse; border: 1px solid #ccc; margin-top: 8px; margin-bottom: 16px;">
        <thead>
            <tr style="background-color: #f5f5f5; border-bottom: 2px solid #ccc; text-align: left;">
                <th style="width: 8%; padding: 8px 6px; font-size: 12px; text-align: left;">Row ID</th>
                <th style="width: 46%; padding: 8px 10px; font-size: 12px; text-align: left;">BEFORE (Prior Step)</th>
                <th style="width: 46%; padding: 8px 10px; font-size: 12px; text-align: left;">AFTER (Modified Step)</th>
            </tr>
        </thead>
        <tbody>
            {''.join(html_rows)}
        </tbody>
    </table>
    '''
    display(HTML(table_html))

# Initialize clean_text column
df['clean_text'] = df['text'].copy()

# Step 2: Remove Lowyat Quotes, BBCode, Signatures, and HTML
Forum-specific noise such as nested quote blocks, spoilers, images, BBCode tags, Lowyat edit signatures, and strict HTML markup are stripped one by one.

### 2.1 Remove Quote Blocks & Quote Headers
Quotes contain text originally written by other users, which distorts the current user's intent. Both `[quote]...[/quote]` blocks and `QUOTE(...)` headers are stripped.

In [ ]:
import re

def remove_quote_blocks(text):
    if not isinstance(text, str): return ""

    # Remove complete BBCode quote blocks: [quote]...[/quote]
    text = re.sub(r'\[quote(?:=[^\]]*)?\][\s\S]*?\[/quote\]', ' ', text, flags=re.IGNORECASE)
    
    # Remove standard stray/closing tags: [quote], [/quote], [quote=...]
    text = re.sub(r'\[/?quote(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    
    # Catch malformed unclosed tags like [/quote without the ']' bracket (Row 3218, Row 63659)
    text = re.sub(r'\[/?quote\b', ' ', text, flags=re.IGNORECASE)
    
    # Remove QUOTE(...) header strings
    text = re.sub(r'QUOTE\s*\([^\)]*?\)', ' ', text, flags=re.IGNORECASE)

    return text

# Apply Step 2.1
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_quote_blocks)
show_step_diff(prev_step, df['clean_text'], "Step 2.1: Remove Quote Blocks & Headers")

### 2.2 Remove Spoilers
Lowyat spoiler blocks `[spoiler]...[/spoiler]` and click-to-show spoiler prompts are stripped.

In [ ]:
def remove_spoilers(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'»\s*Click to show Spoiler.*?«', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[spoiler(?:=[^\]]*)?\][\s\S]*?\[/spoiler\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?spoiler(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    return text

# Apply Step 2.2
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_spoilers)
show_step_diff(prev_step, df['clean_text'], "Step 2.2: Remove Spoilers")

### 2.3 Remove Images & Code Blocks
BBCode image tags `[IMG]...[/IMG]` and code blocks `[code]...[/code]` are stripped.

In [ ]:
def remove_images_and_code(text):
    if not isinstance(text, str):
        return ""
    
    # Full Image blocks: catches [IMG]...[/IMG], CODE[IMG]...[/IMG], [/IMG]...[/IMG], and img]...[/img]
    text = re.sub(r'(?:CODE\s*)?\[?/?IMG\][\s\S]*?\[/?IMG\]', ' ', text, flags=re.IGNORECASE)
    
    # Code blocks: [code]...[/code]
    text = re.sub(r'\[code(?:=[^\]]*)?\][\s\S]*?\[/code\]', ' ', text, flags=re.IGNORECASE)
    
    # Stray / standalone tags: [img], [/img], img], [code], [/code]
    text = re.sub(r'\[?/?img\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?code(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    
    # Stray CODE[ prefixes
    text = re.sub(r'\bCODE\[', ' ', text, flags=re.IGNORECASE)

    # Stray targets uppercase 'CODE' when immediately followed by a non-space character
    text = re.sub(r'\bCODE(?=[^\s])', ' ', text)
    
    return text

# Apply Step 2.3
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_images_and_code)
show_step_diff(prev_step, df['clean_text'], "Step 2.3: Remove Images & Code Blocks")

### 2.4 Remove Video, Attachments & Links BBCode
Strips BBCode wrappers for YouTube, embedded videos, Lowyat attachment IDs, and email tags.

In [ ]:
def remove_media_bbcode(text):
    if not isinstance(text, str): 
        return ""
    
    # YouTube & Video complete blocks (handles malformed brackets like YOUTUBE]...[/YOUTUBE)
    text = re.sub(r'\[?YOUTUBE(?:=[^\]\n]*)?\]?[\s\S]*?\[/?YOUTUBE\]?', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[?VIDEO\s*(?:=[^\]\n]*)?\]?[\s\S]*?\[/?VIDEO\s*\]?', ' ', text, flags=re.IGNORECASE)
    
    # Standalone / Unclosed YouTube & Video tags
    text = re.sub(r'\[/?(?:youtube|video)\s*(?:=[^\]\n]*)?\]?', ' ', text, flags=re.IGNORECASE)
    
    # URL & Linkz tags (handles complete tags and unclosed tags like `[/url` missing `]`)
    text = re.sub(r'\[/?url(?:=[^\]\n]*)?\]?', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?linkz(?:=[^\]\n]*)?\]?', ' ', text, flags=re.IGNORECASE)
    
    # Attachments & Attachment IDs
    text = re.sub(r'\[attachmentid\s*=\s*\d+\]?', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[/?attachment(?:=[^\]\n]*)?\]?', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'Attached\s+(?:thumbnail|image|file|picture)\(s\)', ' ', text, flags=re.IGNORECASE)
    
    # Email tags
    text = re.sub(r'\[/?email(?:=[^\]\n]*)?\]?', ' ', text, flags=re.IGNORECASE)
    
    return text

# Apply Step 2.4
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_media_bbcode)
show_step_diff(prev_step, df['clean_text'], "Step 2.4: Remove Media & Attachment BBCode")

### 2.5 Remove BBCode Text Formatting Tags
Strips bold `[b]`, italic `[i]`, underline `[u]`, strikethrough `[s]`, color `[color]`, size `[size]`, and font `[font]` tags while retaining inner content words.

In [ ]:
def remove_formatting_tags(text):
    if not isinstance(text, str):
        return ""
    
    # Multi-character tags: [color=...], [size=...], [font=...]
    text = re.sub(r'\[/?(?:color|size|font)(?:=[^\]]*)?\]', ' ', text, flags=re.IGNORECASE)
    
    # Closing tags are ALWAYS BBCode (cleans [/b]Chicken without skipping)
    text = re.sub(r'\[/(?:b|i|u|s)\]', ' ', text, flags=re.IGNORECASE)
    
    # Opening tags: [b], [i], [u] are stripped; [s] only stripped if not an editorial prefix like '[s]ince'
    text = re.sub(r'\[(?:b|i|u)\]', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'\[s\](?![a-z])', ' ', text, flags=re.IGNORECASE)
    
    # Malformed tag typos like ]b], ]i]
    text = re.sub(r'\][biuses]\]', ' ', text, flags=re.IGNORECASE)
    
    return text

# Apply Step 2.5
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_formatting_tags)
show_step_diff(prev_step, df['clean_text'], "Step 2.5: Remove Text Formatting Tags")

### 2.6 Remove Lowyat Edit Signatures & Moderator Notices
Strips auto-generated trailing lines like `This post has been edited by...` and moderator redactions `<...removed...>`.

In [ ]:
def remove_signatures_and_redactions(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'This post has been edited by.*?(?=\n|$)', ' ', text, flags=re.IGNORECASE)
    text = re.sub(r'<[a-zA-Z0-9\s_\-\/]+removed[a-zA-Z0-9\s_\-]*>', ' ', text, flags=re.IGNORECASE)
    return text

# Apply Step 2.6
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_signatures_and_redactions)
show_step_diff(prev_step, df['clean_text'], "Step 2.6: Remove Signatures & Moderator Notices")

### 2.7 Remove Strict HTML Tags
Removes leftover raw HTML elements (`<br>`, `<a>`, `<div>`, `<p>`, `<span>`, `<table>`, etc.).

In [ ]:
def remove_html_tags(text):
    if not isinstance(text, str):
        return ""
    
    # Remove complete iframe, script, and style blocks
    text = re.sub(r'<(?:iframe|script|style)\b[^>]*>[\s\S]*?</(?:iframe|script|style)>', ' ', text, flags=re.IGNORECASE)
    
    # Remove standard HTML tags (avoids single-letter race/slang tags like <C>, <M>, <I>, <S>)
    text = re.sub(r'</?(?:div|br|p|span|table|tbody|tr|td|th|thead|tfoot|ul|ol|li|img|a|strong|em|hr|blockquote|font|button|input|form)\b[^>]*>', ' ', text, flags=re.IGNORECASE)
    
    return text

# Apply Step 2.7
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_html_tags)
show_step_diff(prev_step, df['clean_text'], "Step 2.7: Remove Strict HTML Tags")

# Step 3: Special Element Masking
Variable numerical, temporal, contact, and hyperlink expressions are mapped into normalized placeholder tokens (`EMAILTOKEN`, `URLTOKEN`, `PHONETOKEN`, `NRICTOKEN`, `PRICETOKEN`, `DATETOKEN`, `TIMETOKEN`) to capture communicative intent without vocabulary fragmentation.

### 3.1 Mask Email Addresses
Replaces email with `EMAILTOKEN`

In [ ]:
def mask_emails(text):
    if not isinstance(text, str):
        return ""
    
    # Standard email address pattern
    text = re.sub(r'\b[A-Za-z0-9._%+\-*]+@[A-Za-z0-9.\-*]+\.[A-Za-z]{2,}\b', ' EMAILTOKEN ', text, flags=re.IGNORECASE)
    
    return text

# Apply Step 3.1
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(mask_emails)
show_step_diff(prev_step, df['clean_text'], "Step 3.1: Mask Emails")

### 3.2 Mask URLs & Web Hyperlinks
Replaces HTTP/HTTPS URLs and `www.` domain links with `URLTOKEN`.

In [ ]:
def mask_urls(text):
    if not isinstance(text, str):
        return ""
    
    # Scheme-based URLs (http, https, hxxp, ftp)
    text = re.sub(r'(?:https?|hxxps?|ftp)://[^\s<>"]+', ' URLTOKEN ', text, flags=re.IGNORECASE)
    
    # www. prefixed links
    text = re.sub(r'(?<![a-zA-Z0-9])www\.[^\s<>"]+', ' URLTOKEN ', text, flags=re.IGNORECASE)
    
    # Known shortlinks / app links with or without paths
    text = re.sub(r'\b(?:t\.me|goo\.gl|bit\.ly|tinyurl\.com|pictr\.com)(?:/[^\s<>"]*)?', ' URLTOKEN ', text, flags=re.IGNORECASE)
    
    # Domain label helper: strictly forbids consecutive dots (ellipses)
    label = r'[a-zA-Z0-9](?:[a-zA-Z0-9-]{0,61}[a-zA-Z0-9])?'
    
    # Malaysian compound domains (.com.my, .edu.my, .gov.my, .org.my, .net.my)
    my_compound = rf'(?<![@a-zA-Z0-9._-])\b(?:{label}\.)+(?:com|edu|gov|org|net)\.my(?:/[^\s<>"]*)?'
    text = re.sub(my_compound, ' URLTOKEN ', text, flags=re.IGNORECASE)
    
    # Standard gTLDs (.com, .org, .net, .edu, .gov, .io, .ai, .app)
    gtlds = rf'(?<![@a-zA-Z0-9._-])\b(?:{label}\.)+(?:com|org|net|edu|gov|io|ai|app)(?:/[^\s<>"]*)?'
    text = re.sub(gtlds, ' URLTOKEN ', text, flags=re.IGNORECASE)
    
    # Generic/short TLDs (e.g. .to, .me, .co) ONLY when followed by an explicit path
    generic_paths = rf'(?<![@a-zA-Z0-9._-])\b(?:{label}\.)+[a-zA-Z]{{2,}}/[^\s<>"]+'
    text = re.sub(generic_paths, ' URLTOKEN ', text, flags=re.IGNORECASE)
    
    return text

# Apply Step 3.2
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(mask_urls)
show_step_diff(prev_step, df['clean_text'], "Step 3.2: Mask URLs")

### 3.3 Mask Phone Numbers
Robustly detects and replaces Malaysian mobile and landline telephone numbers, as well as international contact numbers (using Google's `phonenumbers` library with strict prefix regex) with `PHONETOKEN`. Large standalone numbers (e.g. 140 million) and alphanumeric reference IDs are protected from false masking.

In [ ]:
from phonenumbers import PhoneNumberMatcher

def mask_phone_numbers(text, default_region="MY"):
    if not isinstance(text, str):
        return ""
    if not text.strip():
        return text
    
    # Parse via Google phonenumbers library
    try:
        matches = list(PhoneNumberMatcher(text, default_region))
        for match in reversed(matches):
            text = text[:match.start] + " PHONETOKEN " + text[match.end:]
    except Exception:
        pass
    
    # Strict Malaysian mobile & landline + international regex (requires explicit phone prefix)
    text = re.sub(r'\b(?:\+?6?01)[0-46-9][-\s]?[0-9]{7,8}\b', ' PHONETOKEN ', text)
    text = re.sub(r'\b(?:\+?6?0[3-9])[-\s]?[0-9]{6,8}\b', ' PHONETOKEN ', text)
    text = re.sub(r'\+\d{1,3}[-\s]?\d{1,4}[-\s]?\d{4,8}\b', ' PHONETOKEN ', text)
    
    # Handle partially redacted / masked numbers using X or asterisks (*)
    text = re.sub(r'(?<!\w)(?:\+?60|0)[1-9][0-9X* -]{6,12}[0-9X*]', ' PHONETOKEN ', text, flags=re.IGNORECASE)
    
    return text

# Apply Step 3.3
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(mask_phone_numbers)
show_step_diff(prev_step, df['clean_text'], "Step 3.3: Mask Phone Numbers")

### 3.4 Mask IC Numbers
Replaces `xxxxxx-xx-xxxx` to  `NRICTOKEN`

In [ ]:
def mask_nric(text):
    if not isinstance(text, str):
        return ""
    if not text.strip():
        return text
    
    # Strictly requires a separator (hyphen or space) between the 6-2-4 blocks.
    # Catches: 990101-14-5678, 990101 14 5678, XXXXXX-XX-6485, XXXXXX XX 6485
    nric_pattern = r'\b[0-9X*]{6}[-\s][0-9X*]{2}[-\s][0-9X*]{4}\b'
    
    text = re.sub(nric_pattern, ' NRICTOKEN ', text, flags=re.IGNORECASE)
    
    return text

# Apply Step 3.4
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(mask_nric)
show_step_diff(prev_step, df['clean_text'], "Step 3.4: Mask IC Numbers")

### 3.5 Mask Prices & Currency Expressions
Detects explicit currency prefixes (`RM`, `MYR`, `$`, `USD`, `SGD`, `RP`, `S$`) and currency suffixes (`sen`, `ringgit`, `dollars`) with context-aware amount masking. General numerical expressions (e.g. `5-star`, `50-10`) are preserved.

In [ ]:
CURRENCY_PREFIX = r'(?:RM|rm|MYR|myr|USD|usd|SGD|sgd|AUD|aud|RP|Rp|S\$|\$)'
CURRENCY_SUFFIX = r'(?:sen|cents?|ringgit|dollars?|myr|rm|usd|sgd|aud|rp)'

PRICE_REGEX = re.compile(
    rf'(?:'
    rf'\b{CURRENCY_PREFIX}\s*\d+(?:[\.,]\d+)?(?:\s*k\b)?|'
    rf'\b\d+(?:[\.,]\d+)?\s*{CURRENCY_SUFFIX}\b'
    rf')',
    re.IGNORECASE
)

# Explicit monetary context for numbers with 'k' (e.g. 'price 7k', 'budget 10k', 'salary 5k')
MONETARY_K_REGEX = re.compile(
    r'\b(?:price|cost|budget|salary|gaji|deposit|refund|discount)\s*(?:is|of|around|about)?\s*(\d+(?:\.\d+)?)\s*k\b',
    re.IGNORECASE
)

def mask_prices(text):
    if not isinstance(text, str):
        return ""
    if not text.strip():
        return text
    text = PRICE_REGEX.sub(' PRICETOKEN ', text)
    text = MONETARY_K_REGEX.sub(' PRICETOKEN ', text)
    return text

# Apply Step 3.5
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(mask_prices)
show_step_diff(prev_step, df['clean_text'], "Step 3.5: Mask Prices & Currencies")

### 3.6 Mask Dates
Replaces numerical calendar dates (`DD/MM/YYYY`, `DD-MM-YY`) and textual month expressions in English and Bahasa Melayu with `DATETOKEN`.

In [ ]:
MONTH_NAMES = (
    r'(?:jan(?:uary)?|januari|'
    r'feb(?:ruary)?|februari|'
    r'mar(?:ch)?|mac|'
    r'apr(?:il)?|'
    r'may|mei|'
    r'jun(?:e)?|'
    r'jul(?:y)?|julai|'
    r'aug(?:ust)?|ogos|'
    r'sep(?:t(?:ember)?)?|'
    r'oct(?:ober)?|okt(?:ober)?|'
    r'nov(?:ember)?|'
    r'dec(?:ember)?|dis(?:ember)?)'
)
DATE_REGEX = re.compile(
    rf'\b(?:0[1-9]|[12][0-9]|3[01])[-/](?:0?[1-9]|1[0-2])(?:[-/](?:\d{{2}}|\d{{4}}))?\b|'
    rf'\b[0-3]?[0-9]\s+{MONTH_NAMES}(?:\s+\d{{2,4}})?\b|'
    rf'\b{MONTH_NAMES}\s+[0-3]?[0-9](?:\s+\d{{2,4}})?\b|'
    r'\b\d{4}[-/][0-1]?[0-9][-/][0-3]?[0-9]\b',
    re.IGNORECASE
)

def mask_dates(text):
    if not isinstance(text, str):
        return ""
    if not text.strip():
        return text
    return DATE_REGEX.sub(' DATETOKEN ', text)

# Apply Step 3.6
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(mask_dates)
show_step_diff(prev_step, df['clean_text'], "Step 3.6: Mask Dates")

### 3.7 Mask Timestamps
Replaces 12-hour and 24-hour time expressions (`HH:MM am/pm`, `HH:MM:SS`, `3.30pm`) with `TIMETOKEN`.

In [ ]:
TIME_REGEX = re.compile(
    r'\b(?:1[0-2]|0?[1-9]):[0-5][0-9](?::[0-5][0-9])?\s?(?:[AaPp][Mm])?\b|'    r'\b(?:[01]?[0-9]|2[0-3]):[0-5][0-9](?::[0-5][0-9])?\b|'    r'\b(?:1[0-2]|0?[1-9])\.[0-5][0-9](?:\.[0-5][0-9])?\s?[AaPp][Mm]\b',    re.IGNORECASE
)

def mask_time(text):
    if not isinstance(text, str):
        return ""
    if not text.strip():
        return text
    return TIME_REGEX.sub(' TIMETOKEN ', text)

# Apply Step 3.7
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(mask_time)
show_step_diff(prev_step, df['clean_text'], "Step 3.7: Mask Timestamps")

# Step 4: Remove Elongated Characters & Repeating Emojis
Normalizes character elongation and repetitive non-ASCII symbols:
1. **Repeating Non-ASCII Characters & Emojis**: Consecutive identical non-ASCII characters and emojis (e.g. `🤣🤣🤣🤣` $\rightarrow$ `🤣`, `🥺🥺` $\rightarrow$ `🥺`) are reduced to 1 occurrence.
2. **Elongated Alphabetical Characters**: Alphabetical characters repeated three or more times are reduced to two occurrences (e.g. `sooooo` $\rightarrow$ `soo`, `waaaaa` $\rightarrow$ `waa`).

In [ ]:
def remove_elongated_content(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    # 1. Reduce repeated non-ASCII characters (including emojis) to 1 occurrence
    text = re.sub(r'([^\x00-\x7F])(?:\s*\1)+', r'\1', text)
    # 2. Reduce elongated alphabetical characters (3 or more -> 2)
    text = re.sub(r'([a-zA-Z])\1{2,}', r'\1\1', text)
    return text

# Apply Step 4
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_elongated_content)
show_step_diff(prev_step, df['clean_text'], "Step 4: Remove Elongated Characters & Repeating Emojis")

# Step 5: Convert Emojis to Text
Converts visual emojis into natural, space-separated English words using `emoji.demojize` and replaces colons and underscores with spaces (e.g. 😂 $\rightarrow$ `face with tears of joy`, 👍 $\rightarrow$ `thumbs up`). This ensures emojis become standard expressive vocabulary instead of compound tokens.

In [ ]:
from emoji import demojize

def convert_emojis(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    # Convert emojis to text and replace underscores/delimiters with spaces for natural words
    converted = demojize(text, delimiters=(" ", " "))
    return converted.replace("_", " ")

# Apply Step 5
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(convert_emojis)
show_step_diff(prev_step, df['clean_text'], "Step 5: Convert Emojis to Text")

# Step 6: Lowercasing
Convert all characters to lowercase to normalize vocabulary.

In [ ]:
def to_lowercase(text):
    return text.lower() if isinstance(text, str) else ""

# Apply Step 6
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(to_lowercase)
show_step_diff(prev_step, df['clean_text'], "Step 6: Lowercasing")

# Step 7: Build Priority-Cascading Slang Dictionary & Normalize Slangs (Hybrid Solution)

Informal Malay, Manglish, and English internet acronyms and shortforms are normalized using a multi-source dictionary cascade with an optimized **Hybrid Architecture**:
- **Multi-Word Phrases** (e.g. `apa khabar`, `hari ini`, `you all`) are normalized using a targeted regex pass.
- **Single-Word Slangs** (e.g. `sbb` $\rightarrow$ `sebab`, `xde` $\rightarrow$ `tak ada`) are normalized using an instantaneous $O(1)$ dictionary lookup, eliminating the overhead of giant 2,000-word regex patterns.

### Slang Dictionary Sources:
1. **`custom_malay_slang.json`**: Custom-curated dictionary for Malaysian forum/SMS shortforms and particle mappings (e.g. `sbb` $\rightarrow$ `sebab`, `xde` $\rightarrow$ `tiada`, `skrg` $\rightarrow$ `sekarang`, `takpe` $\rightarrow$ `tidak apa`, `lhdn`).
2. **`custom_english_slang.json`**: Custom-curated dictionary for modern forum banter and internet expressions (e.g. `topkek` $\rightarrow$ `laughing out loud`, `bye` $\rightarrow$ `goodbye`, `fyi` $\rightarrow$ `for your information`, `idk` $\rightarrow$ `i do not know`, `tbh` $\rightarrow$ `to be honest`).
3. **`malay_slangdict.json`**: Open-source Malay slang dataset from **Mendeley Data** ([Mendeley Dataset Resource](https://data.mendeley.com/datasets/mgv2n2vcb9/3/files/a7b86a2f-1175-4ff0-b813-d95218534cd4)) for broad Bahasa Melayu informal and regional|| spelling variations.
4. **`english_slangdict.json`**: Pre-built Internet Slang dictionary from **Ekphrasis / NoSlang** ([Ekphrasis SlangDict Source](https://github.com/cbaziotis/ekphrasis/blob/master/ekphrasis/dicts/noslang/slangdict.py)) for global English chat acronyms (`asap`, `brb`, `lol`, `rofl`).

### Priority Order Rationale:
Dictionaries are loaded in order: `custom_malay_slang.json` $\rightarrow$ `custom_english_slang.json` $\rightarrow$ `malay_slangdict.json` $\rightarrow$ `english_slangdict.json`.
If a shortform appears in multiple dictionaries, the earlier entry is preserved.

In [ ]:
import json

def build_slang_dictionary(file_paths):
    combined_dict = {}
    for path in file_paths:
        try:
            with open(path, 'r', encoding='utf-8') as f:
                d = json.load(f)
                for k, v in d.items():
                    clean_k = str(k).strip().lower()
                    clean_v = str(v).strip().lower()
                    if clean_k and clean_k not in combined_dict:
                        combined_dict[clean_k] = clean_v
        except FileNotFoundError:
            pass
    return combined_dict

# Priority-cascaded slang dictionary list
slang_file_priority = [
    'custom_malay_slang.json',
    'custom_english_slang.json',
    'malay_slangdict.json',
    'english_slangdict.json'
    ]
MASTER_SLANG_DICT = build_slang_dictionary(slang_file_priority)

# Separate into multi-word phrases and single-word slangs for optimal hybrid processing
MULTI_WORD_SLANG = {k: v for k, v in MASTER_SLANG_DICT.items() if ' ' in k or '-' in k}
SINGLE_WORD_SLANG = {k: v for k, v in MASTER_SLANG_DICT.items() if ' ' not in k and '-' not in k}

printmd(f"**Total Master Slang Entries Compiled:** `{len(MASTER_SLANG_DICT):,}` entries (Single Words: `{len(SINGLE_WORD_SLANG):,}`, Phrases: `{len(MULTI_WORD_SLANG):,}`)")

# Print the complete compiled master slang dictionary
printmd("#### Full Master Slang Dictionary Mapping:")
display(pd.DataFrame(list(MASTER_SLANG_DICT.items()), columns=["Slang Key", "Normalized Replacement"]))

# Small compiled regex strictly for multi-word phrases
if MULTI_WORD_SLANG:
    phrase_keys = [re.escape(k) for k in sorted(MULTI_WORD_SLANG.keys(), key=len, reverse=True)]
    PHRASE_REGEX = re.compile(r'\b(' + '|'.join(phrase_keys) + r')\b', re.IGNORECASE)
else:
    PHRASE_REGEX = None
    
def normalize_slangs(text):
    if not isinstance(text, str) or not text.strip():
        return ""
    # 1. Multi-word phrase replacement (targeted small regex)
    if PHRASE_REGEX:
        text = PHRASE_REGEX.sub(lambda m: MULTI_WORD_SLANG[m.group(0).lower()], text)
    # 2. Fast O(1) dictionary word lookup (eliminates giant regex overhead)
    words = text.split()
    return ' '.join([SINGLE_WORD_SLANG.get(w.lower(), w) for w in words])

# Apply Step 7
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(normalize_slangs)
show_step_diff(prev_step, df['clean_text'], "Step 7: Slang Normalization (Hybrid)")

# Step 8: Fix Contractions
English contractions are expanded using the `contractions` library.

In [ ]:
from contractions import fix as fix_contractions_func

def fix_contractions(text):
    return fix_contractions_func(text) if isinstance(text, str) else ""

# Apply Step 8
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(fix_contractions)
show_step_diff(prev_step, df['clean_text'], "Step 8: Fix Contractions")

### Demonstration of Contraction Expansion on Words Without Apostrophes
Casual internet and forum users frequently omit apostrophes (e.g. typing `dont`, `cant`, `didnt`, `im`, `thats`). The `contractions` library automatically recognizes these informal words and expands them into standard tokens without needing manual apostrophe insertion.

In [ ]:
# Filter rows that had NO apostrophes in the prior step but were successfully expanded by contractions
no_apostrophe_mask = (prev_step != df['clean_text']) & (~prev_step.astype(str).str.contains(r"['’]|&#x27;", regex=True))

# Render visual colored diff table for apostrophe-less expansions
show_step_diff(prev_step[no_apostrophe_mask], df['clean_text'][no_apostrophe_mask], "Demonstration: Contraction Expansion on Words Without Apostrophes")

# Step 9: Remove All Punctuations
All punctuation marks are stripped and replaced with spaces, ensuring zero punctuation noise survives into downstream models.

In [ ]:
import string

def remove_punctuations(text):
    if not isinstance(text, str): return ""
    punct_pattern = f"[{re.escape(string.punctuation)}]"
    return re.sub(punct_pattern, ' ', text)

# Apply Step 9
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_punctuations)
show_step_diff(prev_step, df['clean_text'], "Step 9: Remove All Punctuations")

# Step 10: Remove Non-Latin Words / Characters
Strips non-ASCII character sequences to focus corpus on Latin-scripted text.

In [ ]:
def remove_non_latin(text):
    if not isinstance(text, str): return ""
    return re.sub(r'[^\x00-\x7F]+', ' ', text)

# Apply Step 10
prev_step = df['clean_text'].copy()
df['clean_text'] = df['clean_text'].apply(remove_non_latin)
show_step_diff(prev_step, df['clean_text'], "Step 10: Remove Non-Latin Characters")

# Step 11: Word Tokenization
Splits cleaned text into individual word tokens using NLTK `word_tokenize`.

In [ ]:
from nltk import download as nltk_download
from nltk.tokenize import word_tokenize

# Download NLTK tokenization models
nltk_download('punkt', quiet=True)
nltk_download('punkt_tab', quiet=True)

def tokenize_words(text):
    if not isinstance(text, str) or not text.strip():
        return []
    return word_tokenize(text)

# Apply Step 11
df['tokens'] = df['clean_text'].apply(tokenize_words)

printmd("**Step 11 Completed:** Tokenized cleaned text into token lists.")
sample_tokens_df = pd.DataFrame({
    "Raw Clean Text": df['clean_text'].head(5),
    "Tokens Output": df['tokens'].head(5)
})
display(sample_tokens_df)

# Step 12: Language Classification and Token Tagging
Tokens are classified using **Sastrawi Malay Lexicon** for Bahasa Melayu and **NLTK Corpus / WordNet / SpaCy** for English, tagging tokens as `TAG`, `MALAY`, `ENGLISH`, or `UNKNOWN` to guide downstream language-specific morphological processing.

In [ ]:
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from nltk.corpus import words, wordnet
from malaya.dictionary import is_malay
import nltk

# Download NLTK dictionary corpora
nltk.download('words', quiet=True)
nltk.download('wordnet', quiet=True)

NLTK_WORDS_SET = {w.lower() for w in words.words()}
MASKING_TAGS = {'emailtoken', 'urltoken', 'phonetoken', 'nrictoken', 'pricetoken', 'timetoken', 'datetoken'}

# Sastrawi standard Malay
stemmer_factory = StemmerFactory()
malay_stemmer = stemmer_factory.create_stemmer()

def is_english_nltk(w):
    return w in NLTK_WORDS_SET or len(wordnet.synsets(w)) > 0
    
def tag_tokens(tokens):
    tagged = []
    for token in tokens:
        token_lower = token.lower()
        if token_lower in MASKING_TAGS:
            tagged.append((token, "TAG"))
        elif is_english_nltk(token_lower):
            tagged.append((token, "ENGLISH"))
        elif is_malay(token_lower):
            tagged.append((token, "MALAY"))
        else:
            tagged.append((token, "UNKNOWN"))
    return tagged

# Apply Step 12
df['tagged_tokens'] = df['tokens'].apply(tag_tokens)

printmd("**Step 12 Completed:** Tagged language and entity categories on tokens via Sastrawi & NLTK.")
sample_tagged_df = pd.DataFrame({
    "Tokens": df['tokens'].head(5),
    "Tagged Tokens": df['tagged_tokens'].head(5)
})
display(sample_tagged_df)

# Step 13: Stop Word Removal
Removes English stopwords (via **SpaCy**) and Malay stopwords (via **Sastrawi & NLTK**), while preserving essential communicative question words (`why`, `how`, `what`, `kenapa`, `mengapa`, `bagaimana`) and stripping isolated single-character noise words.

In [ ]:
from spacy.lang.en.stop_words import STOP_WORDS as SPACY_STOPWORDS
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Essential inquiry keywords preserved to maintain strong feature signals for Inquiry intent
PRESERVED_WORDS = {'why', 'how', 'what', 'which', 'where', 'who', 'kenapa', 'mengapa', 'bagaimana'}

ENGLISH_STOPWORDS = set(SPACY_STOPWORDS) - PRESERVED_WORDS
sastrawi_stop_factory = StopWordRemoverFactory()
MALAY_STOPWORDS = set(sastrawi_stop_factory.get_stop_words()) - PRESERVED_WORDS

# Display active Stopword sets
printmd(f"#### Active Stopword Catalogs (Preserved: `{', '.join(sorted(PRESERVED_WORDS))}`):")
printmd(f"**English Stopwords ({len(ENGLISH_STOPWORDS)} words via SpaCy):**\n`{', '.join(sorted(ENGLISH_STOPWORDS))}`")
printmd(f"**Malay Stopwords ({len(MALAY_STOPWORDS)} words via Sastrawi):**\n`{', '.join(sorted(MALAY_STOPWORDS))}`")

def remove_stopwords(tagged_tokens):
    cleaned = []
    for word, tag in tagged_tokens:
        word_lower = word.lower()
        if tag == "ENGLISH" and word_lower in ENGLISH_STOPWORDS:
            continue
        elif tag == "MALAY" and word_lower in MALAY_STOPWORDS:
            continue
        # Also drop single-letter noise words (e.g. isolated 'i')
        if len(word) > 1 or word.isdigit() or word_lower in MASKING_TAGS:
            cleaned.append((word, tag))
    return cleaned

# Apply Step 13
df['filtered_tokens'] = df['tagged_tokens'].apply(remove_stopwords)
printmd("**Step 13 Completed:** Filtered out stopwords and single-character noise.")
sample_stopwords_df = pd.DataFrame({
    "Before (Tagged Tokens)": df['tagged_tokens'].head(5),
    "After (Filtered Tokens)": df['filtered_tokens'].head(5)
})
display(sample_stopwords_df)

# Step 14: Part-of-Speech (POS) Tagging for English Tokens
NLTK's `pos_tag` assigns contextual Penn Treebank grammatical tags (e.g. Verbs `VB/VBD/VBG`, Nouns `NN/NNS`, Adjectives `JJ`, Adverbs `RB`) to English tokens, which are mapped to WordNet POS categories to ensure accurate lemmatization.

In [ ]:
from nltk import pos_tag

# Download NLTK POS tagger models
nltk_download('averaged_perceptron_tagger', quiet=True)
nltk_download('averaged_perceptron_tagger_eng', quiet=True)

# Helper to convert Penn Treebank POS tag to WordNet POS category
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

def pos_tag_english_tokens(tagged_tokens):
    words_list = [t[0] for t in tagged_tokens]
    if not words_list:
        return []
    
    # Run NLTK pos_tag on token list
    raw_pos = pos_tag(words_list)
    
    pos_tagged_list = []
    for (word, lang_tag), (_, ptb_tag) in zip(tagged_tokens, raw_pos):
        if lang_tag == "ENGLISH":
            wn_pos = get_wordnet_pos(ptb_tag)
            pos_tagged_list.append((word, lang_tag, wn_pos, ptb_tag))
        else:
            pos_tagged_list.append((word, lang_tag, None, "N/A"))
            
    return pos_tagged_list

# Apply Step 14
df['pos_tagged_tokens'] = df['filtered_tokens'].apply(pos_tag_english_tokens)
printmd("**Step 14 Completed:** Assigned contextual POS tags to English tokens via NLTK.")
sample_pos_df = pd.DataFrame({
    "Before (Filtered Tokens)": df['filtered_tokens'].head(5),
    "After (POS Tagged Tokens)": df['pos_tagged_tokens'].head(5)
})
display(sample_pos_df)

# Step 15: Morphological Normalization (Lemmatization with POS & Stemming)
English tokens are lemmatized using **WordNetLemmatizer** with their contextual POS tags; Malay tokens are stemmed using **Sastrawi**. The resulting token list is flattened into the final `clean_text` corpus.

In [ ]:
from nltk.stem import WordNetLemmatizer
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

nltk_download('wordnet', quiet=True)

lemmatizer = WordNetLemmatizer()
stemmer_factory = StemmerFactory()
malay_stemmer = stemmer_factory.create_stemmer()
def lemmatize_and_stem(pos_tagged_tokens):
    processed = []
    for word, lang_tag, wn_pos, _ in pos_tagged_tokens:
        if lang_tag == "ENGLISH":
            # Lemmatize using contextual WordNet POS tag
            processed.append((lemmatizer.lemmatize(word, pos=wn_pos), lang_tag))
        elif lang_tag == "MALAY":
            processed.append((malay_stemmer.stem(word), lang_tag))
        else:
            processed.append((word, lang_tag))
    return processed

# Apply Step 15
df['morph_tokens'] = df['pos_tagged_tokens'].apply(lemmatize_and_stem)
df['clean_text'] = df['morph_tokens'].apply(lambda t_list: ' '.join([t[0] for t in t_list]))

printmd("**Step 15 Completed:** Normalized morphology (POS-guided lemmatization and stemming).")
sample_morph_df = pd.DataFrame({
    "Before (POS Tagged Tokens)": df['pos_tagged_tokens'].head(5),
    "After (Lemmatized & Stemmed Tokens)": df['morph_tokens'].head(5),
    "Reconstructed Clean Text": df['clean_text'].head(5)
})
display(sample_morph_df)

# Step 16: Filter Out Empty & 0-Token Posts
Posts that contained exclusively quotes, spoilers, stripped media, or stopwords are reduced to 0 tokens / empty strings. Filtering them out prevents zero-vector noise from degrading TF-IDF feature weights and classifier decision boundaries.

In [ ]:
initial_total_rows = len(df)

# Filter for posts containing at least 1 valid cleaned token
df = df[df['clean_text'].astype(str).str.strip().str.len() > 0].reset_index(drop=True)
dropped_rows = initial_total_rows - len(df)

printmd(f"**Step 16 Completed:** Dropped `{dropped_rows:,}` empty/0-token posts ({dropped_rows/initial_total_rows*100:.2f}% of total).")
printmd(f"**Cleaned Dataset for Model Training:** `{len(df):,}` valid informative posts remaining.")

# Step 17: Train / Test Dataset Splitting
The preprocessed corpus is partitioned into a **70% training set** and a **30% testing set** using stratified multi-label sampling.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df[target_cols], test_size=0.30, random_state=42
)

printmd(f"**Dataset Partition:** Training set: `{len(X_train):,}` samples ({len(X_train)/len(df)*100:.1f}%), Testing set: `{len(X_test):,}` samples ({len(X_test)/len(df)*100:.1f}%)")
printmd("#### Class Distribution Across Partitions:")
split_summary_df = pd.DataFrame({
    "Training Set Count": y_train.sum(),
    "Testing Set Count": y_test.sum(),
    "Total Count": df[target_cols].sum()
})
display(split_summary_df)

# Step 18: TF-IDF Vectorization
Converts text into numerical feature vectors by weighting word frequency against dataset rarity:

$$\text{TF-IDF} = \text{TF} \times \text{IDF}$$

- **TF (Term Frequency)**: How often a word appears in this post.
- **IDF (Inverse Document Frequency)**: How rare the word is across the entire dataset (gives higher weight to unique intent keywords like `refund`, `scam`, `why`).

In [ ]:
from joblib import dump as joblib_dump
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit TF-IDF Vectorizer on Training Data and Transform Test Data
tfidf = TfidfVectorizer(stop_words=None, min_df=3, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Persist TF-IDF vectorizer artifact
joblib_dump(tfidf, "tfidf_vectorizer.joblib")
feature_names = tfidf.get_feature_names_out()
printmd(f"**TF-IDF Vocabulary Space:** Fitted `{len(feature_names):,}` unique n-gram features across `{X_train_tfidf.shape[0]:,}` training documents.")

# Preview First 5 Documents in TF-IDF Matrix Table (Sample of Active Non-Zero Features)
sample_tfidf_array = X_train_tfidf[:5].toarray()
active_indices = np.where(sample_tfidf_array.sum(axis=0) > 0)[0][:15] # first 15 active features
active_features = [feature_names[i] for i in active_indices]

sample_tfidf_df = pd.DataFrame(
    sample_tfidf_array[:, active_indices],
    index=[f"Doc {i} (Row {X_train.index[i]})" for i in range(5)],
    columns=active_features
)

printmd("#### TF-IDF Feature Matrix Preview (First 5 Training Documents & Sample Active Words):")
display(sample_tfidf_df.style.format("{:.4f}"))

# Top TF-IDF Weighted Terms per Document (First 5 Training Posts)
top_terms_summary = []
for i in range(5):
    row_vec = sample_tfidf_array[i]
    top_word_indices = np.argsort(row_vec)[-5:][::-1]
    top_word_indices = [idx for idx in top_word_indices if row_vec[idx] > 0]
    top_keywords = ", ".join([f"{feature_names[idx]} ({row_vec[idx]:.3f})" for idx in top_word_indices])
    
    top_terms_summary.append({
        "Document ID": f"Doc {i} (Row {X_train.index[i]})",
        "Cleaned Text Snippet": X_train.iloc[i][:70] + ("..." if len(X_train.iloc[i]) > 70 else ""),
        "Highest TF-IDF Feature Weights": top_keywords
    })

printmd("#### Top TF-IDF Weighted Keywords in First 5 Training Documents:")
display(pd.DataFrame(top_terms_summary))

# Step 19: Train One-vs-Rest Logistic Regression Model
Train One-vs-Rest Logistic Regression with balanced class weights to handle class imbalance across all 6 communicative intent categories.

In [ ]:
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression

lr_model = OneVsRestClassifier(LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42))
lr_model.fit(X_train_tfidf, y_train)

# Persist Logistic Regression model artifact
joblib_dump(lr_model, "logistic_regression_model.joblib")
printmd("**Logistic Regression Model:** Successfully trained and saved to `logistic_regression_model.joblib`.")

# Step 20: Logistic Regression Feature Interpretation & 1D Sigmoid Probability Curves
Inspect highest positive coefficient weights ($w_j$) and plot the 1D Sigmoid curves $\sigma(z) = \frac{1}{1 + e^{-z}}$ across **all 6 communicative intent categories**.

### 20.1 Top Predictive Features per Intent Class (Logistic Regression)
Words with the strongest positive coefficients ($w_j$) that significantly increase the predicted probability of the intent being present.

In [ ]:
feature_names = tfidf.get_feature_names_out()
top_words_lr = {}

for i, label in enumerate(target_cols):
    coefs = lr_model.estimators_[i].coef_[0]
    top_indices = np.argsort(coefs)[-8:][::-1]
    top_words_lr[label] = [f"{feature_names[idx]} ({coefs[idx]:.2f})" for idx in top_indices]
    
printmd("#### Top 8 Predictive Features per Intent Class (Logistic Regression):")
display(pd.DataFrame(top_words_lr))

### 20.2 1D Sigmoid Probability Curves Across All 6 Intent Categories
Visualizes how Logistic Regression maps raw linear decision scores $z = \mathbf{w}^T \mathbf{x} + b$ into continuous probabilities $P(Y=1) = \sigma(z) = \frac{1}{1 + e^{-z}}$ with the decision threshold at $P = 0.5$ ($z = 0$).

In [ ]:
# 1D Sigmoid Probability Curves across all 6 Intent Categories
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
axes = axes.ravel()

# Sample test posts for clean, uncluttered visual display
sample_size = min(300, X_test_tfidf.shape[0])
sample_idx = np.random.RandomState(42).choice(X_test_tfidf.shape[0], size=sample_size, replace=False)
X_test_sample = X_test_tfidf[sample_idx]

for i, label in enumerate(target_cols):
    # Calculate linear decision score z for each post
    z_scores = lr_model.estimators_[i].decision_function(X_test_sample)
    y_true_sample = y_test[label].iloc[sample_idx].values
    
    # Generate continuous smooth Sigmoid curve
    z_min, z_max = z_scores.min() - 0.5, z_scores.max() + 0.5
    z_line = np.linspace(z_min, z_max, 300)
    p_line = 1 / (1 + np.exp(-z_line))
    
    # Plot Sigmoid S-Curve
    axes[i].plot(z_line, p_line, color="#1f77b4", linewidth=2.5, label=r"$\sigma(z) = \frac{1}{1 + e^{-z}}$")
    
    # Plot Decision Threshold line at P = 0.5 and vertical line at z = 0
    axes[i].axhline(0.5, color="#d62728", linestyle="--", linewidth=1.5, label="Threshold ($P=0.5$)")
    axes[i].axvline(0.0, color="#7f7f7f", linestyle=":", linewidth=1.2)
    
    # Scatter sample test posts (y=0 at bottom, y=1 at top with slight vertical jitter)
    jitter = np.random.RandomState(42).uniform(-0.02, 0.02, size=len(z_scores))
    neg_mask = (y_true_sample == 0)
    pos_mask = (y_true_sample == 1)
    
    axes[i].scatter(z_scores[neg_mask], np.zeros(np.sum(neg_mask)) + jitter[neg_mask], 
                    color="#4C72B0", alpha=0.6, edgecolors='k', s=30, label=f"No {label} (0)")
    axes[i].scatter(z_scores[pos_mask], np.ones(np.sum(pos_mask)) + jitter[pos_mask], 
                    color="#2ca02c", alpha=0.75, edgecolors='k', s=40, marker='^', label=f"{label} (1)")
    
    axes[i].set_title(f"Logistic Regression: {label} (Sigmoid Curve)", fontsize=11, fontweight='bold')
    axes[i].set_xlabel(r"Linear Score $z = \mathbf{w}^T \mathbf{x} + b$", fontsize=9)
    axes[i].set_ylabel(r"Predicted Probability $P(Y=1)$", fontsize=9)
    axes[i].set_ylim(-0.08, 1.08)
    axes[i].legend(loc="lower right", fontsize=8)
    axes[i].grid(True, linestyle=":", alpha=0.5)

plt.suptitle("Logistic Regression 1D Sigmoid Probability Curves Across All 6 Intent Categories", fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

# Step 21: Logistic Regression Decision Boundary Equations & Model Evaluation
Calculates the linear decision boundary equations and evaluates test set predictions using Confusion Matrices and the custom Classification Report.

### 21.1 Logistic Regression Linear Decision Boundary Equations
Calculates the linear decision score for each intent category:

$$z = \text{intercept} + (w_1 \times \text{word}_1) + (w_2 \times \text{word}_2) + \dots$$

If $z > 0$ ($P > 0.5$), the model predicts that the intent is **Present (1)**.

In [ ]:
from sklearn.metrics import confusion_matrix

# Custom Multi-Label Classification Report Generator (Accuracy, Precision, Recall, F1-Score)
def calculate_classification_metrics(y_true, y_pred, target_names):
    report_rows = []
    acc_list, prec_list, rec_list, f1_list = [], [], [], []
    
    for i, col in enumerate(target_names):
        y_t = y_true[col].values if isinstance(y_true, pd.DataFrame) else y_true[:, i]
        y_p = y_pred[:, i] if isinstance(y_pred, np.ndarray) else y_pred[col].values
        
        tp = int(np.sum((y_t == 1) & (y_p == 1)))
        tn = int(np.sum((y_t == 0) & (y_p == 0)))
        fp = int(np.sum((y_t == 0) & (y_p == 1)))
        fn = int(np.sum((y_t == 1) & (y_p == 0)))
        
        acc = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
        prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        rec = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0.0
        
        acc_list.append(acc)
        prec_list.append(prec)
        rec_list.append(rec)
        f1_list.append(f1)
        
        report_rows.append({
            'Intent Category': col,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-Score': f1
        })
        
    # Append Macro Average Row
    report_rows.append({
        'Intent Category': 'MACRO AVERAGE',
        'Accuracy': float(np.mean(acc_list)),
        'Precision': float(np.mean(prec_list)),
        'Recall': float(np.mean(rec_list)),
        'F1-Score': float(np.mean(f1_list))
    })
    
    report_df = pd.DataFrame(report_rows)
    return report_df, acc_list, prec_list, rec_list, f1_list

printmd("#### Logistic Regression Linear Decision Boundary Equations:")
for i, label in enumerate(target_cols):
    coefs = lr_model.estimators_[i].coef_[0]
    intercept = lr_model.estimators_[i].intercept_[0]
    top_3_idx = np.argsort(coefs)[-3:][::-1]
    terms = " + ".join([f"({coefs[idx]:.2f} * {feature_names[idx]})" for idx in top_3_idx])
    print(f"[{label:<11}] z = {intercept:.2f} + {terms} + ...")

### 21.2 Confusion Matrices for Logistic Regression
Evaluates True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN) per communicative intent category.

In [ ]:
y_pred_lr = lr_model.predict(X_test_tfidf)

# Calculate Custom Classification Report & Metric Lists
lr_report_df, lr_accuracies, lr_precisions, lr_recalls, lr_f1_scores = calculate_classification_metrics(
    y_test, y_pred_lr, target_cols
)

# Plot Confusion Matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

for i, label in enumerate(target_cols):
    cm = confusion_matrix(y_test[label], y_pred_lr[:, i])
    sns.heatmap(cm, ax=axes[i], annot=True, fmt="d", cmap="Blues",
                xticklabels=[f"Pred No {label}", f"Pred {label}"],
                yticklabels=[f"Act No {label}", f"Act {label}"])
    axes[i].set_title(f"Logistic Regression: {label} (F1: {lr_f1_scores[i]:.3f})", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

### 21.3 Custom Classification Report for Logistic Regression
Displays the exact calculated **Accuracy**, **Precision**, **Recall**, and **F1-Score** alongside the **Macro Average**.

In [ ]:
printmd("#### Classification Report for Logistic Regression:")
display(lr_report_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}'
}))

# Step 22: Train One-vs-Rest Linear Support Vector Classifier (LinearSVC) Model
Train One-vs-Rest Linear Support Vector Classifier (`LinearSVC`) with balanced class weights to maximize the geometric margin across all 6 communicative intent categories.

In [ ]:
from sklearn.svm import LinearSVC

svc_model = OneVsRestClassifier(LinearSVC(class_weight='balanced', max_iter=2000, random_state=42))
svc_model.fit(X_train_tfidf, y_train)

# Persist Linear SVC model artifact
joblib_dump(svc_model, "linear_svc_model.joblib")
printmd("**Linear SVC Model:** Successfully trained and saved to `linear_svc_model.joblib`.")

# Step 23: Linear SVC Feature Interpretation, Support Vector Calculation & Margin Ramp Response

### 1. Linear SVC Decision Function Formulation:
Linear SVC computes the scalar decision score $z$ (signed distance to the separating hyperplane) for each comment vector $\mathbf{x}$:

$$z = \mathbf{w}^T \mathbf{x} + b = \sum_{j=1}^n w_j x_j + b$$

### 2. Functional Margin & Support Vector Identification:
For binary classification with labels mapped to $y_i \in \{-1, +1\}$, the **Functional Margin** is defined as $y_i (\mathbf{w}^T \mathbf{x}_i + b)$.
A comment $\mathbf{x}_i$ is identified as a **Support Vector** (or margin violator) if and only if it satisfies the margin condition:

$$y_i (\mathbf{w}^T \mathbf{x}_i + b) \le 1 \quad \iff \quad \text{Slack Variable } \xi_i = \max(0, 1 - y_i z_i) \ge 0$$

- **$y_i z_i > 1$**: Correctly classified outside the margin buffer (high confidence, non-support vector).
- **$0 \le y_i z_i \le 1$**: Sits directly on or inside the **Margin Buffer $[-1, +1]$** (Support Vector holding up the boundary).
- **$y_i z_i < 0$**: Margin violation / misclassified post ($z_i$ has the wrong sign).

### 3. Linear SVC Clamped Margin Ramp Response:
Linear SVC saturates to class $-1.0$ when $z \le -1$ and class $+1.0$ when $z \ge +1$, ramping linearly across the **Margin Buffer $[-1, +1]$**:

$$f_{\text{SVC}}(z) = \text{clip}(z, -1, 1) = \begin{cases} -1.0 & \text{if } z < -1 \\ z & \text{if } -1 \le z \le 1 \\ +1.0 & \text{if } z > 1 \end{cases}$$

### 23.1 Top Predictive Linear Weights per Intent Category
Words with the highest positive coefficient weights ($w_j$) that strongly push predictions above the decision threshold.

In [ ]:
top_words_svc = {}
for i, label in enumerate(target_cols):
    coefs = svc_model.estimators_[i].coef_[0]
    top_indices = np.argsort(coefs)[-8:][::-1]
    top_words_svc[label] = [f"{feature_names[idx]} ({coefs[idx]:.2f})" for idx in top_indices]
    
printmd("#### Linear SVC Top 8 Predictive Features per Intent Class:")
display(pd.DataFrame(top_words_svc))

### 23.2 Support Vectors Calculation & Margin Statistics
We calculate the exact functional margin $y_i z_i$ for all test comments and quantify the total number of **Support Vectors** (posts lying within or violating the margin buffer $y_i z_i \le 1$) across all 6 intent categories.

In [ ]:
# Support Vectors Calculation across all 6 Intent Categories
sv_summary = []
total_test_samples = X_test_tfidf.shape[0]

for i, label in enumerate(target_cols):
    z_scores = svc_model.estimators_[i].decision_function(X_test_tfidf)
    y_binary = y_test[label].values if isinstance(y_test, pd.DataFrame) else y_test[:, i]
    
    # Convert binary {0, 1} to {-1, +1} for functional margin evaluation
    y_pm = np.where(y_binary == 1, 1.0, -1.0)
    functional_margins = y_pm * z_scores
    
    # Support Vectors: functional margin <= 1.0 (on margin boundary, inside buffer, or misclassified)
    sv_mask = functional_margins <= 1.0
    sv_count = int(np.sum(sv_mask))
    
    # Breakdown: Inside Margin vs Misclassified (Violations)
    on_or_inside_margin = int(np.sum((functional_margins >= 0) & (functional_margins <= 1.0)))
    misclassified = int(np.sum(functional_margins < 0))
    
    sv_summary.append({
        'Intent Category': label,
        'Total Test Posts': total_test_samples,
        'Total Support Vectors': sv_count,
        'Support Vector Ratio (%)': f"{sv_count / total_test_samples * 100:.2f}%",
        'Inside Margin Buffer [0, 1]': on_or_inside_margin,
        'Margin Violations (z < 0)': misclassified
    })

sv_df = pd.DataFrame(sv_summary)
printmd("#### Support Vectors Quantification per Intent Category (Formula: $y_i z_i \le 1$):")
display(sv_df)

### 23.3 Linear SVC Margin Ramp & Support Vector Buffer Response Plot
Visualizes the **Linear SVC Margin Ramp Function** $f(z) = \text{clip}(z, -1, 1)$ across the scalar decision score $z = \mathbf{w}^T \mathbf{x} + b$:
- **Green Line**: Linear SVC Clamped Margin Response ($f(z) = \text{clip}(z, -1, 1)$)
- **Grey Shaded Region**: SVC Margin Buffer Zone $[-1, +1]$ (where Support Vectors lie)
- **Decision Boundaries**: Negative Boundary at $z = -1$, Separating Hyperplane at $z = 0$, Positive Boundary at $z = +1$

In [ ]:
# Plot Linear SVC Margin Ramp Function & Margin Buffer Zone
z_range = np.linspace(-4, 4, 500)
svc_ramp = np.clip(z_range, -1.0, 1.0)

plt.figure(figsize=(12, 5.5))

# 1. Highlight SVC Margin Buffer [-1, +1]
plt.axvspan(-1, 1, color='#dcdcdc', alpha=0.9, label='SVC Margin Buffer [-1, +1]')

# 2. Plot Linear SVC Clamped Margin Ramp Curve (Green)
plt.plot(z_range, svc_ramp, color='#2ca02c', linewidth=3.2, label='Linear SVC (Margin Ramp)')

# 3. Reference Coordinate Axes
plt.axhline(0, color='#888888', linestyle='-', linewidth=0.9)
plt.axvline(0, color='#666666', linestyle='-', linewidth=1.1, label='Separating Hyperplane ($z=0$)')
plt.axvline(-1, color='#888888', linestyle='--', linewidth=1.2, label='Negative Margin ($z=-1$)')
plt.axvline(1, color='#888888', linestyle='--', linewidth=1.2, label='Positive Margin ($z=+1$)')

# 4. Highlight Saturated Anchor Point at z = -4
plt.plot(-4, -1.0, 'o', color='#2ca02c', markersize=9)
plt.axvline(-4, color='#444444', linestyle=':', linewidth=1.4)

plt.title("Linear SVC: Scalar Decision Score ($z$) vs Prediction Output & Margin Buffer", fontsize=12, fontweight='bold')
plt.xlabel(r"Scalar Score ($z = \mathbf{w}^T \mathbf{x} + b$) $rightarrow$", fontsize=11)
plt.ylabel(r"Prediction Output $\uparrow$", fontsize=11)
plt.xlim(-4.2, 4.2)
plt.ylim(-1.25, 1.25)
plt.yticks([-1.0, -0.5, 0.0, 0.5, 1.0])
plt.xticks([-4, -3, -2, -1, 0, 1, 2, 3, 4])
plt.legend(loc='lower left', bbox_to_anchor=(0.0, -0.25), ncol=4, frameon=False, fontsize=9.5)
plt.grid(True, linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()

# Step 24: Linear SVC Model Evaluation & Custom Classification Report
We evaluate the Linear SVC model using Confusion Matrix heatmaps and generate the custom Classification Report table.

### 24.1 Confusion Matrices for Linear SVC
Evaluates True Positives (TP), True Negatives (TN), False Positives (FP), and False Negatives (FN) per communicative intent category.

In [ ]:
y_pred_svc = svc_model.predict(X_test_tfidf)

# Calculate Custom Classification Report & Metric Lists for Linear SVC
svc_report_df, svc_accuracies, svc_precisions, svc_recalls, svc_f1_scores = calculate_classification_metrics(
    y_test, y_pred_svc, target_cols
)

# Plot Confusion Matrices
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.ravel()

for i, label in enumerate(target_cols):
    cm = confusion_matrix(y_test[label], y_pred_svc[:, i])
    sns.heatmap(cm, ax=axes[i], annot=True, fmt="d", cmap="Purples",
                xticklabels=[f"Pred No {label}", f"Pred {label}"],
                yticklabels=[f"Act No {label}", f"Act {label}"])
    axes[i].set_title(f"Linear SVC: {label} (F1: {svc_f1_scores[i]:.3f})", fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

### 24.2 Custom Classification Report for Linear SVC
Displays the exact calculated **Accuracy**, **Precision**, **Recall**, and **F1-Score** alongside the **Macro Average**.

In [ ]:
printmd("#### Classification Report for Linear SVC:")
display(svc_report_df.style.format({
    'Accuracy': '{:.4f}',
    'Precision': '{:.4f}',
    'Recall': '{:.4f}',
    'F1-Score': '{:.4f}'
}))

# Step 25: Comprehensive All-Metrics Model Comparison
Consolidated comparison of **Accuracy**, **Precision**, **Recall**, and **F1-Score** between Logistic Regression and Linear SVC across all 6 intent categories + Macro Average.

In [ ]:
# Consolidated All-Metrics Comparison Table
comp_table = pd.DataFrame({
    'Category': target_cols,
    'LR Accuracy': lr_accuracies,
    'SVC Accuracy': svc_accuracies,
    'LR Precision': lr_precisions,
    'SVC Precision': svc_precisions,
    'LR Recall': lr_recalls,
    'SVC Recall': svc_recalls,
    'LR F1-Score': lr_f1_scores,
    'SVC F1-Score': svc_f1_scores,
})

# Add Macro Average Row
comp_table.loc[len(comp_table)] = {
    'Category': 'MACRO AVERAGE',
    'LR Accuracy': np.mean(lr_accuracies),
    'SVC Accuracy': np.mean(svc_accuracies),
    'LR Precision': np.mean(lr_precisions),
    'SVC Precision': np.mean(svc_precisions),
    'LR Recall': np.mean(lr_recalls),
    'SVC Recall': np.mean(svc_recalls),
    'LR F1-Score': np.mean(lr_f1_scores),
    'SVC F1-Score': np.mean(svc_f1_scores),
}

printmd("### Overall Model Performance Comparison (Logistic Regression vs Linear SVC)")
display(comp_table.style.format({col: "{:.3f}" for col in comp_table.columns if col != 'Category'}))

# Side-by-Side Multi-Metric Visual Comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
metrics_to_plot = [
    ("Accuracy", "LR Accuracy", "SVC Accuracy"),
    ("Precision", "LR Precision", "SVC Precision"),
    ("Recall", "LR Recall", "SVC Recall"),
    ("F1-Score", "LR F1-Score", "SVC F1-Score")
]

x = np.arange(len(target_cols))
width = 0.35

for idx, (m_name, lr_c, svc_c) in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    ax.bar(x - width/2, comp_table.iloc[:6][lr_c], width, label="Logistic Regression", color="#4C72B0")
    ax.bar(x + width/2, comp_table.iloc[:6][svc_c], width, label="Linear SVC", color="#8172B2")
    ax.set_title(f"Model Comparison: {m_name} per Category", fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(target_cols, rotation=25)
    ax.set_ylim(0, 1.05)
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# Step 26: Preprocessing Pipeline Encapsulation & Model Artifact Loading
We encapsulate Steps 2 through 15 into a clean standalone `preprocess_pipeline(raw_text)` function and load the saved `.joblib` model artifacts from disk for deployment.

In [ ]:
from joblib import load as joblib_load

# Encapsulate Steps 2 to 15 in exact execution sequence
def preprocess_pipeline(raw_text):
    # Step 2: Remove Lowyat Quotes, BBCode, Signatures, and HTML
    text = remove_quote_blocks(raw_text)
    text = remove_spoilers(text)
    text = remove_images_and_code(text)
    text = remove_media_bbcode(text)
    text = remove_formatting_tags(text)
    text = remove_signatures_and_redactions(text)
    text = remove_html_tags(text)
    
    # Step 3: Special Element Masking (7 masking functions in order)
    text = mask_emails(text)
    text = mask_urls(text)
    text = mask_phone_numbers(text)
    text = mask_nric(text)
    text = mask_prices(text)
    text = mask_dates(text)
    text = mask_time(text)
    
    # Step 4: Remove Elongated Characters & Repeating Emojis
    text = remove_elongated_content(text)
    
    # Step 5: Convert Emojis to Text
    text = convert_emojis(text)
    
    # Step 6: Lowercasing
    text = to_lowercase(text)
    
    # Step 7: Slang Normalization (Hybrid)
    text = normalize_slangs(text)
    
    # Step 8: Fix Contractions
    text = fix_contractions(text)
    
    # Step 9: Remove All Punctuations
    text = remove_punctuations(text)
    
    # Step 10: Remove Non-Latin Characters
    text = remove_non_latin(text)
    
    # Step 11: Word Tokenization
    tokens = tokenize_words(text)
    
    # Step 12: Language Classification and Token Tagging
    tagged = tag_tokens(tokens)
    
    # Step 13: Stop Word Removal
    filtered = remove_stopwords(tagged)
    
    # Step 14: Part-of-Speech (POS) Tagging for English Tokens
    pos_tagged = pos_tag_english_tokens(filtered)
    
    # Step 15: Morphological Normalization (Lemmatization with POS & Stemming)
    morph_tokens = lemmatize_and_stem(pos_tagged)
    
    return ' '.join([t[0] for t in morph_tokens])

# Load serialized artifacts from disk
loaded_tfidf = joblib_load("tfidf_vectorizer.joblib")
loaded_lr = joblib_load("logistic_regression_model.joblib")
loaded_svc = joblib_load("linear_svc_model.joblib")
printmd("**Artifacts Status:** All serialized models (`TF-IDF`, `Logistic Regression`, `Linear SVC`) loaded successfully from disk.")

# Step 27: Real-Time Prediction on Unseen Posts (Showing Both Models)
Execute real-time multi-label classification on brand-new unseen forum posts, displaying predictions simultaneously from **both Logistic Regression and Linear SVC**.

In [ ]:
def predict_unseen(raw_text):
    cleaned_input = preprocess_pipeline(raw_text)
    
    if not cleaned_input.strip():
        return {
            "raw_text": raw_text,
            "cleaned_text": cleaned_input,
            "lr_intents": ["Neutral/None"],
            "svc_intents": ["Neutral/None"]
        }
    
    vectorized_input = loaded_tfidf.transform([cleaned_input])
    
    # 1. Logistic Regression Prediction
    lr_pred_array = loaded_lr.predict(vectorized_input)[0]
    lr_intents = [target_cols[i] for i, val in enumerate(lr_pred_array) if val == 1]
    
    # 2. Linear SVC Prediction
    svc_pred_array = loaded_svc.predict(vectorized_input)[0]
    svc_intents = [target_cols[i] for i, val in enumerate(svc_pred_array) if val == 1]
    
    return {
        "raw_text": raw_text,
        "cleaned_text": cleaned_input,
        "lr_intents": lr_intents if lr_intents else ["Neutral/None"],
        "svc_intents": svc_intents if svc_intents else ["Neutral/None"]
    }

# 1. Automated Test Verification on Sample Forum Comments
test_samples = [
    "QUOTE(seller @ 10am) barang rosak teruk refund RM50 https://shop.com/scam please help!!!",
    "bila tarikh release movie baru tu? nak book ticket kat shopee",
    "i",
    "may i know which course you chosing"
]

printmd("#### Automated Verification on Test Sample Posts:")
for post in test_samples:
    result = predict_unseen(post)
    print(f"Post        : {result['raw_text']}")
    print(f"Cleaned     : {result['cleaned_text']}")
    print(f"Logistic Reg: {result['lr_intents']}")
    print(f"Linear SVC  : {result['svc_intents']}\n")

# 2. Interactive Input While-Loop (Exits only on 'q')
def interactive_intent_predictor():
    printmd("### Interactive Forum Comment Intent Predictor")
    printmd("Type any forum comment below to test intent classification across **both models** (or enter '**q**' to quit):")
    
    while True:
        user_post = input("\nYour comment (or 'q' to quit): ").strip()
        if user_post.lower() == 'q':
            printmd("**Exiting interactive session.**")
            break
        if not user_post:
            print("Please enter a non-empty comment.")
            continue
            
        res = predict_unseen(user_post)
        printmd("#### Prediction Output:")
        printmd(f"- **Raw Input:** `{res['raw_text']}`")
        printmd(f"- **Cleaned Input:** `{res['cleaned_text']}`")
        printmd(f"- **Logistic Regression Intents:** **{', '.join(res['lr_intents'])}**")
        printmd(f"- **Linear SVC Intents:** **{', '.join(res['svc_intents'])}**")

# Run interactive predictor
interactive_intent_predictor()